In [3]:
!pip install pyngrok

In [4]:
from pyngrok import ngrok
ngrok.set_auth_token("33vuVQJmxXVpqAoEcgFwV0RC4WA_58TYsADphsGzAUsCGymyK")


In [5]:
!pip install flask pyngrok pandas plotly


In [6]:
import sqlite3
from pathlib import Path

DB_PATH = "finance.db"

# Delete existing DB for fresh start
if Path(DB_PATH).exists():
    Path(DB_PATH).unlink()
    print("Previous database deleted. Starting fresh.")

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
cur = conn.cursor()

# Create tables
cur.executescript("""
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS User (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    monthly_income REAL NOT NULL
);

CREATE TABLE IF NOT EXISTS Food (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    expense_date TEXT DEFAULT (datetime('now','localtime')),
    amount REAL NOT NULL,
    description TEXT
);

CREATE TABLE IF NOT EXISTS Rent (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    expense_date TEXT DEFAULT (datetime('now','localtime')),
    amount REAL NOT NULL,
    description TEXT
);

CREATE TABLE IF NOT EXISTS Travel (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    expense_date TEXT DEFAULT (datetime('now','localtime')),
    amount REAL NOT NULL,
    description TEXT
);

CREATE TABLE IF NOT EXISTS Miscellaneous (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    expense_date TEXT DEFAULT (datetime('now','localtime')),
    amount REAL NOT NULL,
    description TEXT
);

CREATE TABLE IF NOT EXISTS Alerts (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    category TEXT,
    expense_id INTEGER,
    amount REAL,
    message TEXT,
    created_at TEXT DEFAULT (datetime('now','localtime'))
);
""")

# Threshold for high expense alerts
THRESHOLD = 10000.0

# Create triggers for category alerts
cur.executescript(f"""
DROP TRIGGER IF EXISTS trg_food_alert;
DROP TRIGGER IF EXISTS trg_rent_alert;
DROP TRIGGER IF EXISTS trg_travel_alert;
DROP TRIGGER IF EXISTS trg_misc_alert;
DROP TRIGGER IF EXISTS trg_total_expense_alert;

CREATE TRIGGER trg_food_alert AFTER INSERT ON Food WHEN NEW.amount > {THRESHOLD}
BEGIN
    INSERT INTO Alerts (category, expense_id, amount, message)
    VALUES ('Food', NEW.id, NEW.amount, 'High expense detected (Food)');
END;

CREATE TRIGGER trg_rent_alert AFTER INSERT ON Rent WHEN NEW.amount > {THRESHOLD}
BEGIN
    INSERT INTO Alerts (category, expense_id, amount, message)
    VALUES ('Rent', NEW.id, NEW.amount, 'High expense detected (Rent)');
END;

CREATE TRIGGER trg_travel_alert AFTER INSERT ON Travel WHEN NEW.amount > {THRESHOLD}
BEGIN
    INSERT INTO Alerts (category, expense_id, amount, message)
    VALUES ('Travel', NEW.id, NEW.amount, 'High expense detected (Travel)');
END;

CREATE TRIGGER trg_misc_alert AFTER INSERT ON Miscellaneous WHEN NEW.amount > {THRESHOLD}
BEGIN
    INSERT INTO Alerts (category, expense_id, amount, message)
    VALUES ('Miscellaneous', NEW.id, NEW.amount, 'High expense detected (Miscellaneous)');
END;

-- Trigger for total expense exceeding income (fires on any category insert)
CREATE TRIGGER trg_total_expense_alert AFTER INSERT ON Food
BEGIN
    INSERT INTO Alerts (category, expense_id, amount, message)
    SELECT 'Total', NULL, SUM(amount), 'Total expenses exceeded income'
    FROM (
        SELECT amount FROM Food
        UNION ALL SELECT amount FROM Rent
        UNION ALL SELECT amount FROM Travel
        UNION ALL SELECT amount FROM Miscellaneous
    )
    WHERE (SELECT monthly_income FROM User ORDER BY id DESC LIMIT 1) <
          (SELECT SUM(amount) FROM (
              SELECT amount FROM Food
              UNION ALL SELECT amount FROM Rent
              UNION ALL SELECT amount FROM Travel
              UNION ALL SELECT amount FROM Miscellaneous
          ));
END;

CREATE TRIGGER trg_total_expense_alert_rent AFTER INSERT ON Rent
BEGIN
    INSERT INTO Alerts (category, expense_id, amount, message)
    SELECT 'Total', NULL, SUM(amount), 'Total expenses exceeded income'
    FROM (
        SELECT amount FROM Food
        UNION ALL SELECT amount FROM Rent
        UNION ALL SELECT amount FROM Travel
        UNION ALL SELECT amount FROM Miscellaneous
    )
    WHERE (SELECT monthly_income FROM User ORDER BY id DESC LIMIT 1) <
          (SELECT SUM(amount) FROM (
              SELECT amount FROM Food
              UNION ALL SELECT amount FROM Rent
              UNION ALL SELECT amount FROM Travel
              UNION ALL SELECT amount FROM Miscellaneous
          ));
END;

CREATE TRIGGER trg_total_expense_alert_travel AFTER INSERT ON Travel
BEGIN
    INSERT INTO Alerts (category, expense_id, amount, message)
    SELECT 'Total', NULL, SUM(amount), 'Total expenses exceeded income'
    FROM (
        SELECT amount FROM Food
        UNION ALL SELECT amount FROM Rent
        UNION ALL SELECT amount FROM Travel
        UNION ALL SELECT amount FROM Miscellaneous
    )
    WHERE (SELECT monthly_income FROM User ORDER BY id DESC LIMIT 1) <
          (SELECT SUM(amount) FROM (
              SELECT amount FROM Food
              UNION ALL SELECT amount FROM Rent
              UNION ALL SELECT amount FROM Travel
              UNION ALL SELECT amount FROM Miscellaneous
          ));
END;

CREATE TRIGGER trg_total_expense_alert_misc AFTER INSERT ON Miscellaneous
BEGIN
    INSERT INTO Alerts (category, expense_id, amount, message)
    SELECT 'Total', NULL, SUM(amount), 'Total expenses exceeded income'
    FROM (
        SELECT amount FROM Food
        UNION ALL SELECT amount FROM Rent
        UNION ALL SELECT amount FROM Travel
        UNION ALL SELECT amount FROM Miscellaneous
    )
    WHERE (SELECT monthly_income FROM User ORDER BY id DESC LIMIT 1) <
          (SELECT SUM(amount) FROM (
              SELECT amount FROM Food
              UNION ALL SELECT amount FROM Rent
              UNION ALL SELECT amount FROM Travel
              UNION ALL SELECT amount FROM Miscellaneous
          ));
END;
""")

conn.commit()
conn.close()
print("✅ Database ready")


✅ Database ready


In [7]:
from flask import Flask, render_template, request, redirect, url_for, flash
import sqlite3
from pathlib import Path
import plotly.graph_objs as go
from plotly.offline import plot

DB_PATH = "finance.db"
CATEGORIES = ['Food','Rent','Travel','Miscellaneous']

def get_conn():
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

app = Flask(__name__)
app.secret_key = "devsecret"

# ---------------- Home Page ----------------
@app.route("/", methods=["GET","POST"])
def home():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT * FROM User ORDER BY id DESC LIMIT 1")
    user = cur.fetchone()

    if request.method == "POST":
        name = request.form.get("name","").strip()
        income = request.form.get("income","").strip()
        if not name or not income:
            flash("Please fill all fields!", "warning")
        else:
            try:
                income_val = float(income)
                cur.execute("INSERT INTO User (name, monthly_income) VALUES (?,?)", (name, income_val))
                conn.commit()
                flash("User details saved!", "success")
                return redirect(url_for("dashboard"))
            except:
                flash("Invalid income value.", "danger")
    conn.close()
    return render_template("home.html", user=user)

# ---------------- Dashboard ----------------
@app.route("/dashboard")
def dashboard():
    conn = get_conn()
    cur = conn.cursor()
    totals = {}
    for cat in CATEGORIES:
        cur.execute(f"SELECT COALESCE(SUM(amount),0) as total FROM {cat}")
        totals[cat] = cur.fetchone()["total"]

    cur.execute("SELECT monthly_income FROM User ORDER BY id DESC LIMIT 1")
    row = cur.fetchone()
    income = row["monthly_income"] if row else 0
    grand_total = sum(totals.values())
    remaining = income - grand_total

    # Bar chart
    fig = go.Figure([go.Bar(x=list(totals.keys()), y=list(totals.values()))])
    fig.update_layout(title="Category-wise Expenses", yaxis_title="Amount (₹)")
    plot_div = plot(fig, output_type='div', include_plotlyjs=False)

    conn.close()
    return render_template("index.html", totals=totals, grand=grand_total, income=income,
                           remaining=remaining, plot_div=plot_div)

# ---------------- Add Expense ----------------
@app.route("/add", methods=["GET","POST"])
def add_expense():
    conn = get_conn()
    cur = conn.cursor()
    if request.method == "POST":
        category = request.form.get("category")
        amount = request.form.get("amount","").strip()
        desc = request.form.get("description","").strip()
        if category not in CATEGORIES:
            flash("Invalid category!", "danger")
            return redirect(url_for("add_expense"))
        try:
            amt = float(amount)
            cur.execute(f"INSERT INTO {category} (amount, description) VALUES (?,?)", (amt, desc))
            conn.commit()
            flash(f"Added ₹{amt:.2f} to {category}", "success")
            conn.close()
            return redirect(url_for("dashboard"))
        except:
            flash("Enter a valid numeric amount.", "warning")
    conn.close()
    return render_template("add.html", categories=CATEGORIES)

# ---------------- Alerts ----------------
@app.route("/alerts")
def alerts():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT * FROM Alerts ORDER BY created_at DESC")
    rows = cur.fetchall()
    conn.close()
    return render_template("alerts.html", alerts=rows)


In [8]:
import os
os.makedirs("templates", exist_ok=True)
print("✅ templates folder ready!")


✅ templates folder ready!


In [9]:
%%writefile templates/base.html
<!doctype html>
<html>
  <head>
    <meta charset="utf-8">
    <title>Expense Tracker</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>

  </head>
  <body class="bg-light">
    <nav class="navbar navbar-expand-lg navbar-light bg-white shadow-sm">
      <div class="container">
        <a class="navbar-brand" href="/">Expense Tracker</a>
        <div>
          <a class="btn btn-outline-primary btn-sm me-2" href="/dashboard">Dashboard</a>
          <a class="btn btn-outline-success btn-sm me-2" href="/add">Add Expense</a>
          <a class="btn btn-outline-warning btn-sm" href="/alerts">Alerts</a>
        </div>
      </div>
    </nav>
    <div class="container py-4">
      {% with messages = get_flashed_messages(with_categories=true) %}
        {% if messages %}
          {% for category, msg in messages %}
            <div class="alert alert-{{ 'success' if category=='success' else 'warning' if category=='warning' else 'danger' }}">{{ msg }}</div>
          {% endfor %}
        {% endif %}
      {% endwith %}
      {% block content %}{% endblock %}
    </div>
  </body>
</html>


Writing templates/base.html


In [10]:
%%writefile templates/home.html
{% extends "base.html" %}
{% block content %}
<h2>Welcome! Enter your details</h2>
<form method="post" class="mt-3">
  <div class="mb-3">
    <label>Name</label>
    <input name="name" class="form-control" placeholder="Your name" required>
  </div>
  <div class="mb-3">
    <label>Monthly Income (₹)</label>
    <input name="income" type="number" step="0.01" class="form-control" placeholder="e.g. 50000" required>
  </div>
  <button class="btn btn-success">Save</button>
</form>
{% endblock %}


Writing templates/home.html


In [11]:
%%writefile templates/index.html
{% extends "base.html" %}
{% block content %}
<h2>Dashboard</h2>
<p>Monthly Income: ₹{{ '%.2f'|format(income) }}</p>
<p>Total Expenses: ₹{{ '%.2f'|format(grand) }}</p>
<p>Remaining: ₹{{ '%.2f'|format(remaining) }}</p>

<div class="mt-4">{{ plot_div|safe }}</div>

<div class="row mt-4">
  {% for k,v in totals.items() %}
    <div class="col-md-3">
      <div class="card mb-3">
        <div class="card-body">
          <h5 class="card-title">{{ k }}</h5>
          <p class="card-text">Total: ₹{{ '%.2f'|format(v) }}</p>
        </div>
      </div>
    </div>
  {% endfor %}
</div>
{% endblock %}


Writing templates/index.html


In [12]:
%%writefile templates/add.html
{% extends "base.html" %}
{% block content %}
<h2>Add Expense</h2>
<form method="post" class="mt-3">
  <div class="mb-3">
    <label>Category</label>
    <select name="category" class="form-select" required>
      {% for cat in categories %}
        <option value="{{ cat }}">{{ cat }}</option>
      {% endfor %}
    </select>
  </div>
  <div class="mb-3">
    <label>Amount (₹)</label>
    <input name="amount" type="number" step="0.01" class="form-control" placeholder="e.g. 250" required>
  </div>
  <div class="mb-3">
    <label>Description (optional)</label>
    <input name="description" class="form-control" placeholder="e.g. lunch">
  </div>
  <button class="btn btn-success">Add</button>
</form>
{% endblock %}


Writing templates/add.html


In [13]:
%%writefile templates/alerts.html
 {% extends "base.html" %}

{% block content %}
<h2>Alerts</h2>

{% if alerts %}
  <table class="table">
    <thead><tr><th>Time</th><th>Category</th><th>Amount</th><th>Message</th></tr></thead>
    <tbody>
      {% for a in alerts %}
        <tr>
          <td>{{ a['created_at'] }}</td>
          <td>{{ a['category'] }}</td>
          <td>{% if a['amount'] %}₹{{ '%.2f'|format(a['amount']) }}{% else %}-{% endif %}</td>
          <td>{{ a['message'] }}</td>
        </tr>
      {% endfor %}
    </tbody>
  </table>
{% else %}
  <p>No alerts yet.</p>
{% endif %}

<!-- Modal for pop-up alert -->
{% if alerts %}
<div class="modal fade" id="alertModal" tabindex="-1" aria-labelledby="alertModalLabel" aria-hidden="true">
  <div class="modal-dialog">
    <div class="modal-content">
      <div class="modal-header">
        <h5 class="modal-title" id="alertModalLabel">New Alert!</h5>
        <button type="button" class="btn-close" data-bs-dismiss="modal" aria-label="Close"></button>
      </div>
      <div class="modal-body">
        {% for a in alerts %}
          <p><strong>{{ a['category'] }}:</strong> ₹{{ '%.2f'|format(a['amount']) if a['amount'] else '-' }} — {{ a['message'] }}</p>
        {% endfor %}
      </div>
      <div class="modal-footer">
        <button type="button" class="btn btn-secondary" data-bs-dismiss="modal">Close</button>
      </div>
    </div>
  </div>
</div>

<script>
  // Show the modal automatically
  var myModal = new bootstrap.Modal(document.getElementById('alertModal'));
  myModal.show();
</script>
{% endif %}

{% endblock %}



Writing templates/alerts.html


In [ ]:
from pyngrok import ngrok

port = 8000
# Make sure to replace YOUR_TOKEN_HERE with a **valid ngrok token**
ngrok.set_auth_token("33vuVQJmxXVpqAoEcgFwV0RC4WA_58TYsADphsGzAUsCGymyK")

public_url = ngrok.connect(port)
print("🌍 Public URL:", public_url)

# Run app
app.run(port=port)


🌍 Public URL: NgrokTunnel: "https://jae-extrusible-debra.ngrok-free.dev" -> "http://localhost:8000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [18/May/2026 17:04:55] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/May/2026 17:04:55] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [18/May/2026 17:05:03] "POST / HTTP/1.1" 302 -
INFO:werkzeug:127.0.0.1 - - [18/May/2026 17:05:05] "GET /dashboard HTTP/1.1" 200 -
